<a href="https://colab.research.google.com/github/incrisvel/adaptaste/blob/main/similarity_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [125]:
%pip -q install pandas numpy scipy scikit-learn plotly

In [126]:
import pandas as pd
import json

# Abre arquivo de reviews
with open('reviews_preprocessed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

preprocessed_reviews_df = pd.DataFrame(data)

In [127]:
preprocessed_reviews_df.head()

,review_id,review_date,place_id,review_text,rating,description_clean,description_tokenized,description_normalized,description_no_stopwords,description_stemmed,description_lemmatized,description_bigrams
0,Ci9DQUlRQUNvZENodHljRjlvT2xaVlRqSXpPRTFSYURKdl...,2026-07-30T17:09:02+00:00,0x6b0a883715bb4c6b:0,Excelente restaurante! A comida é muito saboro...,5.0,Excelente restaurante! A comida é muito saboro...,"[Excelente, restaurante, !, A, comida, é, muit...","[excelente, restaurante, a, comida, é, muito, ...","[excelente, restaurante, comida, muito, saboro...","[excel, restaurant, com, muit, sabor, bem, ser...","[excelente, restaurante, comer, muito, saboros...","[[excelente, restaurante], [restaurante, comer..."
1,Ci9DQUlRQUNvZENodHljRjlvT20xVFVVaHpaRVV4VVZWSl...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Fui pela primeira vez e gostei bastante! A com...,5.0,Fui pela primeira vez e gostei bastante! A com...,"[Fui, pela, primeira, vez, e, gostei, bastante...","[fui, pela, primeira, vez, e, gostei, bastante...","[primeira, vez, gostei, bastante, comida, muit...","[prim, vez, gost, bast, com, muit, sabor, peg,...","[primeiro, vez, gostar, bastante, comer, muito...","[[primeiro, vez], [vez, gostar], [gostar, bast..."
2,Ci9DQUlRQUNvZENodHljRjlvT2psSmFEVkVNak5NTnpONV...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Ambiente simples e gostoso familiar comida óti...,5.0,Ambiente simples e gostoso familiar comida óti...,"[Ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, gostoso, familiar, comida,...","[ambi, simpl, gost, famili, com, ótim, gost, m...","[ambiente, simples, gostoso, familiar, comer, ...","[[ambiente, simples], [simples, gostoso], [gos..."
3,ChZDSUhNMG9nS0VJQ0FnSUNHMWZEb1NBEAE,2022-08-30T17:10:53+00:00,0x6b0a883715bb4c6b:0,"Se é qualidade que procura, não tem outra igua...",5.0,"Se é qualidade que procura, não tem outra igua...","[Se, é, qualidade, que, procura, ,, não, tem, ...","[se, é, qualidade, que, procura, não, tem, out...","[qualidade, procura, não, outra, igual, agilid...","[qual, proc, não, outr, igual, agil, entreg, t...","[qualidade, procurar, não, outro, igual, agili...","[[qualidade, procurar], [procurar, não], [não,..."
4,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB,2022-08-30T17:10:52+00:00,0x6b0a883715bb4c6b:0,"Comida maravilhosa. Super caseira, bem tempera...",5.0,"Comida maravilhosa. Super caseira, bem tempera...","[Comida, maravilhosa, ., Super, caseira, ,, be...","[comida, maravilhosa, super, caseira, bem, tem...","[comida, maravilhosa, super, caseira, bem, tem...","[com, maravilh, sup, cas, bem, temper, tend, s...","[comida, maravilhoso, super, caseira, bem, tem...","[[comida, maravilhoso], [maravilhoso, super], ..."


# Bag of Words

In [128]:
from sklearn.feature_extraction.text import CountVectorizer

# Extrai todos os tokens de todas as reviews
tokens_column = preprocessed_reviews_df["description_no_stopwords"]
tokens = tokens_column.tolist()

bow_vectorizer = CountVectorizer(
    analyzer=lambda tokens: tokens
)
bow_matrix = bow_vectorizer.fit_transform(
    tokens_column
)

# Recupera o vocabulário
bow_words = bow_vectorizer.get_feature_names_out()

# Transforma a matriz em DataFrame
df_bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_words
)

# Adicionar o ID para conectar com a base original
df_bow.insert(0, "review_text", preprocessed_reviews_df["review_text"].values)

# Visualizar
df_bow.head()

,review_text,aaaaamei,abacaxi,abaixo,abendessen,abençoada,abençoe,aber,abertas,aberto,...,ótima,ótimas,ótimo,ótimos,última,últimas,último,única,único,über
0,Excelente restaurante! A comida é muito saboro...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Fui pela primeira vez e gostei bastante! A com...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Ambiente simples e gostoso familiar comida óti...,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,"Se é qualidade que procura, não tem outra igua...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,"Comida maravilhosa. Super caseira, bem tempera...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# TF-IDF

In [129]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criar o modelo TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer=lambda tokens: tokens
)

# Criar matriz
tfidf_matrix = tfidf_vectorizer.fit_transform(
    preprocessed_reviews_df["description_no_stopwords"]
)

# Recuperar o vocabulário
tfidf_words = tfidf_vectorizer.get_feature_names_out()

# Transformar matriz em DF
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_words
)

df_tfidf.insert(0, "review_text", preprocessed_reviews_df["review_text"].values)

df_tfidf.head(20)

,review_text,aaaaamei,abacaxi,abaixo,abendessen,abençoada,abençoe,aber,abertas,aberto,...,ótima,ótimas,ótimo,ótimos,última,últimas,último,única,único,über
0,Excelente restaurante! A comida é muito saboro...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,Fui pela primeira vez e gostei bastante! A com...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,Ambiente simples e gostoso familiar comida óti...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.151180,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,"Se é qualidade que procura, não tem outra igua...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,"Comida maravilhosa. Super caseira, bem tempera...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
5,As palavras para descrever a comida é simplesm...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
6,"A comida deve ser boa pelas indicações mas, o ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
7,Excelente comida e serviço,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
8,Top,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
9,Não sei na outras empresas mas já foi servido...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


# Similaridade entre textos

In [130]:
from sklearn.metrics.pairwise import cosine_similarity

In [131]:
# Calcula similaridade a partir do BoW
similarity_bow = cosine_similarity(
    df_bow.drop(columns="review_text")
)

dt_sim_bow = pd.DataFrame(
    similarity_bow,
    index=preprocessed_reviews_df["review_text"],
    columns=preprocessed_reviews_df["review_text"]
)

dt_sim_bow.head()

review_text,"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨","Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!","Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau","As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo.\nA próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.","A comida deve ser boa pelas indicações mas, o telefone é de enfeite pois ninguém atende, nem mesmo após 20 ligações em um espaço de 30 minutos!!",Excelente comida e serviço,Top,"Não sei na outras empresas mas já foi servido frango crú ou duro , macarrão branco, comida sem sal algum a feijoada não e boa.",...,muito saborosa,Excelente!,Sensacional,A melhor da região ❤️❤️,Excelente,Perfeito ❤️🌹❤️ …,A melhor de todas! Recomendo,Tudo ótimo e atendimento excelente!,Marmitas deliciosas 😋🤩 …,Recomendo!!! Super deliciosas!!\n😋 😋 …
review_text,,,,,,,,,,,,,,,,,,,,,
"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨",1.000000,0.227110,0.183533,0.223607,0.127920,0.300000,0.051640,0.230940,0.0,0.044721,...,0.424264,0.2,0.0,0.0,0.2,0.0,0.000000,0.200000,0.0,0.000000
"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",0.227110,1.000000,0.186081,0.108821,0.138343,0.189258,0.041885,0.093659,0.0,0.072548,...,0.229416,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.093659
Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,0.183533,0.186081,1.000000,0.153897,0.097823,0.191180,0.059235,0.132453,0.0,0.051299,...,0.162221,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000
"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!",0.223607,0.108821,0.153897,1.000000,0.000000,0.372678,0.000000,0.000000,0.0,0.100000,...,0.316228,0.0,0.0,0.0,0.0,0.0,0.000000,0.111803,0.0,0.000000
"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",0.127920,0.138343,0.097823,0.000000,1.000000,0.106600,0.110096,0.246183,0.0,0.095346,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.123091,0.000000,0.0,0.369274


In [132]:
# Calcula similaridade a partir do TF-IDF
similarity_tfidf = cosine_similarity(
    df_tfidf.drop(columns="review_text")
)

dt_sim_tfidf = pd.DataFrame(
    similarity_tfidf,
    index=preprocessed_reviews_df["review_text"],
    columns=preprocessed_reviews_df["review_text"]
)

dt_sim_tfidf.head()

review_text,"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨","Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!","Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau","As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo.\nA próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.","A comida deve ser boa pelas indicações mas, o telefone é de enfeite pois ninguém atende, nem mesmo após 20 ligações em um espaço de 30 minutos!!",Excelente comida e serviço,Top,"Não sei na outras empresas mas já foi servido frango crú ou duro , macarrão branco, comida sem sal algum a feijoada não e boa.",...,muito saborosa,Excelente!,Sensacional,A melhor da região ❤️❤️,Excelente,Perfeito ❤️🌹❤️ …,A melhor de todas! Recomendo,Tudo ótimo e atendimento excelente!,Marmitas deliciosas 😋🤩 …,Recomendo!!! Super deliciosas!!\n😋 😋 …
review_text,,,,,,,,,,,,,,,,,,,,,
"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨",1.000000,0.101567,0.046544,0.074669,0.041217,0.123324,0.008559,0.087817,0.0,0.007912,...,0.261253,0.122276,0.0,0.0,0.122276,0.0,0.000000,0.101837,0.0,0.000000
"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",0.101567,1.000000,0.080314,0.035230,0.045367,0.080605,0.005258,0.020171,0.0,0.024956,...,0.125176,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.052753
Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,0.046544,0.080314,1.000000,0.063365,0.015287,0.055853,0.007381,0.028316,0.0,0.006823,...,0.049590,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000
"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!",0.074669,0.035230,0.063365,1.000000,0.000000,0.185659,0.000000,0.000000,0.0,0.049793,...,0.117975,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.107273,0.0,0.000000
"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",0.041217,0.045367,0.015287,0.000000,1.000000,0.031610,0.016993,0.065189,0.0,0.015709,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.070189,0.000

# Heatmap de similaridade

In [133]:
import plotly.graph_objects as go

texts = preprocessed_reviews_df["review_text"]

def create_heatmap(matrix, title):
  custom_data = []

  for i in range(len(texts)):
    line = []

    for j in range(len(texts)):
      line.append([
          texts[i],
          texts[j]
      ])

    custom_data.append(line)

  fig = go.Figure(
      data=go.Heatmap(
          z=matrix,
          x=texts,
          y=texts,
          colorscale="Blues",
          zmin=0,
          zmax=1,
          customdata=custom_data,
          hovertemplate=
            "<b>%{customdata[0]}</b><br>"
                "<b>%{customdata[1]}</b><br>"
                "<b>Similaridade:</b> %{z:.3f}"
                "<extra></extra>"
      )
  )

  fig.update_layout(
      title=title,
      xaxis_title="Review",
      yaxis_title="Review",
      width=900,
      height=850
  )

  fig.show()

In [ ]:
create_heatmap(
    similarity_bow,
    "Similaridade entre reviews — Bag of Words"
)

In [ ]:
create_heatmap(
    similarity_tfidf,
    "Similaridade entre reviews — TF-IDF"
)

# Clustering

In [241]:
from sklearn.cluster import KMeans
import numpy as np
from sklearn.metrics import silhouette_score

# Número de amostras
n_samples = tfidf_matrix.shape[0]

# Valores de K testados
k_values = [
    k for k in range(2, min(31, n_samples))
]

results = []
models = {}

# Criar e ajustar o modelo
for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
        max_iter=300
    )

    labels = model.fit_predict(tfidf_matrix)
    n_labels = len(np.unique(labels))

    if 1 < n_labels < n_samples:
        silhouette = silhouette_score(
            tfidf_matrix,
            labels,
            metric="cosine",
            sample_size=min(2000, n_samples),
            random_state=42
        )
    else:
        silhouette = np.nan

    results.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette_cosine": silhouette,
        "cluster_sizes": np.bincount(labels).tolist()
    })

    models[k] = model

# Adicionar o resultado a um dataframe
metrics_df = pd.DataFrame(results)

In [242]:
import plotly.express as px

# Visualizar as métricas
fig = px.line(
    metrics_df,
    x="k",
    y="inertia",
    markers=True,
    title="Método do cotovelo (TF-IDF + K-Means)",
    labels={
        "k": "Número de clusters (K)",
        "inertia": "Inércia"
    }
)

fig.show()

In [243]:
display(
    metrics_df.sort_values(
        "silhouette_cosine",
        ascending=False
    )
)

,k,inertia,silhouette_cosine,cluster_sizes
25,27,1533.038360,0.206882,"[51, 72, 144, 101, 623, 67, 55, 35, 50, 28, 11..."
27,29,1516.106427,0.206838,"[16, 96, 106, 80, 26, 55, 129, 52, 66, 35, 795..."
26,28,1519.811247,0.201577,"[9, 39, 58, 138, 49, 69, 100, 79, 105, 18, 74,..."
22,24,1553.486281,0.200766,"[26, 137, 126, 73, 97, 42, 44, 23, 56, 81, 98,..."
28,30,1528.247850,0.199795,"[30, 128, 63, 101, 53, 793, 10, 124, 3, 94, 35..."
16,18,1614.098513,0.193443,"[17, 69, 104, 102, 86, 78, 89, 69, 82, 23, 36,..."
23,25,1546.174181,0.192367,"[18, 34, 68, 39, 138, 124, 100, 17, 81, 51, 70..."
21,23,1575.351504,0.192361,"[101, 102, 18, 39, 44, 136, 35, 60, 86, 67, 94..."
19,21,1595.492017,0.190148,"[1, 45, 103, 100, 101, 13, 17, 67, 69, 74, 105..."
17,19,1600.535605,0.189572,"[52, 212, 90, 101, 139, 97, 52, 40, 68, 76, 49..."


In [302]:
# Selecionar o melhor modelo

N_CLUSTERS = 15
kmeans = models[N_CLUSTERS]
preprocessed_reviews_df["cluster"] = kmeans.labels_

# Análise de clustering


## Verificação manual


In [305]:
# Verificar 10 amostras de cada cluster
for cluster_id in sorted(preprocessed_reviews_df["cluster"].unique()):
    cluster_df = preprocessed_reviews_df[preprocessed_reviews_df["cluster"] == cluster_id]

    print(f"\n{'=' * 100}")
    print(f"CLUSTER {cluster_id} — {len(cluster_df)} reviews")
    print("=" * 100)

    columns_to_show = [
        col for col in [
            "place_id",
            "rating",
            "review_text"
        ]
        if col in cluster_df.columns
    ]

    display(cluster_df[columns_to_show].head(10))


CLUSTER 0 — 264 reviews


,place_id,rating,review_text
0,0x6b0a883715bb4c6b:0,5.0,Excelente restaurante! A comida é muito saboro...
3,0x6b0a883715bb4c6b:0,5.0,"Se é qualidade que procura, não tem outra igua..."
4,0x6b0a883715bb4c6b:0,5.0,"Comida maravilhosa. Super caseira, bem tempera..."
5,0x6b0a883715bb4c6b:0,5.0,As palavras para descrever a comida é simplesm...
19,0x94df1891f8b8e1fd:0,5.0,"Comida caseira, bem feita. Lugar tranquilo, li..."
20,0x94df1891f8b8e1fd:0,5.0,Sou frequentador assiduo. Super recomendo. Um ...
21,0x94df1891f8b8e1fd:0,5.0,Muito bom! Me surpreendi bastante com a qualid...
22,0x94df1891f8b8e1fd:0,5.0,"Restaurante incrível para dia de semana, ambie..."
30,0x94df18bfe8d9f45f:0,1.0,Buffet livre de domingo muito fraco e incompat...
33,0x94df18bfe8d9f45f:0,5.0,"Comida fenomenal, diversos tipos de carnes, pe..."



CLUSTER 1 — 997 reviews


,place_id,rating,review_text
1,0x6b0a883715bb4c6b:0,5.0,Fui pela primeira vez e gostei bastante! A com...
2,0x6b0a883715bb4c6b:0,5.0,Ambiente simples e gostoso familiar comida óti...
6,0x94defd80e5de4f0f:0,3.0,"A comida deve ser boa pelas indicações mas, o ..."
9,0x94defd80e5de4f0f:0,2.0,Não sei na outras empresas mas já foi servido...
10,0x94defd80e5de4f0f:0,5.0,Funcionários são.muito bem servidos profission...
11,0x94defd80e5de4f0f:0,1.0,Não conheço.
13,0x94defd80e5de4f0f:0,5.0,Delícias...
14,0x94defd80e5de4f0f:0,5.0,"Cozinha industrial, ótimos benefícios para sua..."
23,0x94df1891f8b8e1fd:0,3.0,"Fomos almoçar as 13:30h, o buffet claro ja est..."
24,0x94df18bfe8d9f45f:0,5.0,"O menu de inverno estava sensacional, pratos i..."



CLUSTER 2 — 40 reviews


...

,place_id,rating,review_text
15,0x94defd80e5de4f0f:0,5.0,Muito boa
32,0x94df18bfe8d9f45f:0,4.0,Comida muito boa
58,0x94df18bfe8d9f45f:0,5.0,Comida muito boa
59,0x94df18bfe8d9f45f:0,1.0,Muito boa
342,0x94df18eac4b2643d:0,5.0,Comida muito boa!!
420,0x94df18eb055b337f:0,5.0,muito boa a comida do\nSabor Imperial
460,0x94df18eb055b337f:0,5.0,Comida muito boa
536,0x94df192ab2980905:0,4.0,Comida muito boa
671,0x94df192acbd036eb:0,5.0,Muito boa.
760,0x94df192b1e932219:0,5.0,Comida muito boa



CLUSTER 13 — 102 reviews


,place_id,rating,review_text
8,0x94defd80e5de4f0f:0,5.0,Top
64,0x94df18bfe8d9f45f:0,5.0,Top 👏👏 …
74,0x94df18bfe8d9f45f:0,5.0,Top
84,0x94df18bfe8d9f45f:0,5.0,Top
97,0x94df18bfe8d9f45f:0,5.0,Top
107,0x94df18bfe8d9f45f:0,5.0,Top
169,0x94df18c387e4e047:0,5.0,Top
175,0x94df18c387e4e047:0,5.0,Atendimento Top!
214,0x94df18c51a132009:0,5.0,Top
240,0x94df18c51a132009:0,5.0,Top recomendo



CLUSTER 14 — 52 reviews


,place_id,rating,review_text
12,0x94defd80e5de4f0f:0,5.0,Local top ótima comida
16,0x94defd80e5de4f0f:0,5.0,Ótima qualidade.
45,0x94df18bfe8d9f45f:0,5.0,Ótima
46,0x94df18bfe8d9f45f:0,5.0,Ótima atendimento
56,0x94df18bfe8d9f45f:0,5.0,Ótima comida
227,0x94df18c51a132009:0,5.0,Ótima comida
326,0x94df18eac4b2643d:0,5.0,Ótima ‼️‼️‼️
352,0x94df18eac4b2643d:0,5.0,Excelente atendimento e a comida é ótima!!!!!
359,0x94df18eac4b2643d:0,4.0,Comida vegana ótima!
360,0x94df18eac4b2643d:0,5.0,Ótima comida vega


## PCA

In [309]:
from sklearn.decomposition import PCA

# Reduzir o TF-IDF para duas dimensões
pca_2d = PCA(n_components=2)

coordinates_2d = pca_2d.fit_transform(
    tfidf_matrix.toarray()
)

# Criar um DataFrame para a visualização
df_pca_2d = pd.DataFrame({
    "id": preprocessed_reviews_df["review_id"],
    "text": preprocessed_reviews_df["review_text"],
    "place": preprocessed_reviews_df["place_id"],
    "cluster": preprocessed_reviews_df["cluster"].astype(str),
    "PCA1": coordinates_2d[:, 0],
    "PCA2": coordinates_2d[:, 1]
})

# Criar o gráfico
fig = px.scatter(
    df_pca_2d,
    x="PCA1",
    y="PCA2",
    color="cluster",
    hover_name="text",
    hover_data={
        "place": True,
        "PCA1": False,
        "PCA2": False
    },
    title="Clusters das reviews — PCA em 2 dimensões"
)

fig.update_traces(
    marker=dict(size=12)
)

fig.update_layout(
    xaxis_title="Componente principal 1",
    yaxis_title="Componente principal 2",
    legend_title="Cluster"
)

fig.show()

In [310]:
# Reduzir o TF-IDF para três dimensões
pca_3d = PCA(n_components=3)

coordinates_3d = pca_3d.fit_transform(
    tfidf_matrix.toarray()
)

# Criar DataFrame para visualização
df_pca_3d = pd.DataFrame({
    "id": preprocessed_reviews_df["review_id"],
    "text": preprocessed_reviews_df["review_text"],
    "place": preprocessed_reviews_df["place_id"],
    "cluster": preprocessed_reviews_df["cluster"].astype(str),
    "PCA1": coordinates_3d[:, 0],
    "PCA2": coordinates_3d[:, 1],
    "PCA3": coordinates_3d[:, 2]
})

# Criar gráfico 3D
fig = px.scatter_3d(
    df_pca_3d,
    x="PCA1",
    y="PCA2",
    z="PCA3",
    color="cluster",
    hover_name="text",
    hover_data={
        "place": True,
        "PCA1": False,
        "PCA2": False,
        "PCA3": False
    },
    title="Clusters das reviews — PCA em 3 dimensões"
)

fig.update_traces(
    marker=dict(size=6)
)

fig.update_layout(
    scene=dict(
        xaxis_title="Componente principal 1",
        yaxis_title="Componente principal 2",
        zaxis_title="Componente principal 3"
    ),
    legend_title="Cluster"
)

fig.show()

In [311]:
print(
    "Variância explicada em 2D:",
    round(pca_2d.explained_variance_ratio_.sum(), 3)
)

print(
    "Variância explicada em 3D:",
    round(pca_3d.explained_variance_ratio_.sum(), 3)
)

Variância explicada em 2D: 0.094
Variância explicada em 3D: 0.131


## Termos representativos

In [312]:
# Centroides calculados pelo K-Means
centroids = kmeans.cluster_centers_

# Quantidade de termos que queremos mostrar
n_terms = 10

results_clusters = []

for cluster_id in range(N_CLUSTERS):

    # Ordenar os termos pelo peso no centroide
    indices = np.argsort(
        centroids[cluster_id]
    )[::-1]

    principal_indices = indices[:n_terms]

    principal_terms = tfidf_words[principal_indices]
    principal_weights = centroids[
        cluster_id,
        principal_indices
    ]

    results_clusters.append({
        "cluster": cluster_id,
        "representative_tokens": ", ".join(principal_terms)
    })

# Criar DataFrame
df_tokens_clusters = pd.DataFrame(results_clusters)

df_tokens_clusters

,cluster,representative_tokens
0,0,"muito, comida, saborosa, bem, bom, gostosa, lu..."
1,1,"comida, recomendo, lugar, não, qualidade, ambi..."
2,2,"deliciosa, comida, atendimento, experiência, s..."
3,3,"melhor, blumenau, restaurante, melhores, cidad..."
4,4,"maravilhoso, lugar, restaurante, tudo, atendim..."
5,5,"atendimento, ótimo, excelente, comida, ambient..."
6,6,"boa, comida, agradável, preço, local, acessíve..."
7,7,"excelente, comida, restaurante, tudo, recomend..."
8,8,"maravilhosa, comida, experiência, atendimento,..."
9,9,"ótimo, atendimento, lugar, local, tudo, almoço..."


In [313]:
# Similaridade entre avaliações e centroides
similarities = cosine_similarity(tfidf_matrix, centroids)

# Número de avaliações retornadas
n_reviews = 10

for cluster_id in sorted(preprocessed_reviews_df["cluster"].unique()):

    # Índices das avaliações pertencentes ao cluster
    cluster_indices = np.where(kmeans.labels_ == cluster_id)[0]

    if len(cluster_indices) == 0:
        continue

    # Similaridade de cada avaliação com o centroide do cluster
    cluster_similarities = similarities[cluster_indices, cluster_id]

    # Ordenar da maior para a menor similaridade ao centroide
    order = np.argsort(cluster_similarities)[::-1]
    ranked_indices = cluster_indices[order]

    selected_indices = []
    selected_texts = []
    selected_vectors = []

    for idx in ranked_indices:

        # Texto normalizado para evitar selecionar duplicatas exatas
        text = str(
            preprocessed_reviews_df.iloc[idx]["review_text"]
        ).strip().lower()

        # Evitar textos duplicados
        if text in selected_texts:
            continue

        candidate_vector = tfidf_matrix[idx]

        # Verificar distância para as avaliações já selecionadas
        if selected_vectors:
            selected_similarity = cosine_similarity(
                candidate_vector,
                tfidf_matrix[selected_indices]
            )[0]

            distances = 1 - selected_similarity

        selected_indices.append(idx)
        selected_texts.append(text)

        if len(selected_indices) >= n_reviews:
            break

    # Montar resultados
    cluster_df = preprocessed_reviews_df.iloc[
        selected_indices
    ].copy()

    cluster_df["similarity_to_centroid"] = similarities[
        selected_indices, cluster_id
    ]

    cluster_df["distance_to_centroid"] = (
        1 - cluster_df["similarity_to_centroid"]
    )

    print(f"\n{'=' * 100}")
    print(f"CLUSTER {cluster_id}")
    print("=" * 100)

    columns_to_show = [
        col for col in [
            "place_id",
            "rating",
            "similarity_to_centroid",
            "distance_to_centroid",
            "review_text"
        ]
        if col in cluster_df.columns
    ]

    display(cluster_df[columns_to_show].reset_index(drop=True))


CLUSTER 0


,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df18eb055b337f:0,5.0,0.745310,0.254690,Comida muito saborosa
1,0x94df192b1e932219:0,5.0,0.745310,0.254690,Comida muito saborosa :)
2,0x94df3b76767a0235:0,5.0,0.636375,0.363625,muito saborosa
3,0x94df1fb4f7fecadb:0,5.0,0.620058,0.379942,Comida muito saborosa e com muito capricho.
4,0x94df1956fa84d84f:0,5.0,0.556411,0.443589,Comida muito saborosa. Atendimento nota 10!!!
5,0x94df192b1e932219:0,5.0,0.551077,0.448923,Lugar muito top com comida saborosa e de quali...
6,0x94df1ff3b120e2db:0,5.0,0.523154,0.476846,Lugar muito agradável.. comida saborosa... val...
7,0x94df1ece1bfad0c7:0,5.0,0.516390,0.483610,"Muito bom, comida fresca e saborosa"
8,0x94df18eac4b2643d:0,5.0,0.514543,0.485457,Comida muito gostosa
9,0x94df3b76767a0235:0,5.0,0.514543,0.485457,Comida muito gostosa!!



CLUSTER 1


,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df1956fa84d84f:0,5.0,0.306225,0.693775,"Excelente lugar, recomendo a todos."
1,0x94df18eb055b337f:0,1.0,0.291170,0.708830,"Minha experiência não foi das melhores, buffet..."
2,0x94df192bbbd5d5bb:0,5.0,0.278424,0.721576,Melhor comida que já tocou meus lábios e que t...
3,0x94df1f7c5b8faea3:0,5.0,0.278331,0.721669,Local incrível! Acolhedor com aquele toque de ...
4,0x94df1ee19d73e25d:0,5.0,0.276999,0.723001,Super recomendo.
5,0x94df1fc9d2166bf1:0,5.0,0.276999,0.723001,Super recomendo!
6,0x94df1fc9d2166bf1:0,5.0,0.276999,0.723001,Super recomendo
7,0x94df1fb06980a08b:0,5.0,0.275405,0.724595,Ótimo ambiente sem contar que é tudo uma delíc...
8,0x94df198ab08528b1:0,5.0,0.268490,0.731510,"Excelente experiência !\nComida deliciosa, lug..."
9,0x94df1f216f80898f:0,5.0,0.262712,0.737288,Gostei da comida. Recomendo.



CLUSTER 2


...

,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df1fccc0cdf215:0,5.0,0.986863,0.013137,Muito boa comida
1,0x94df1fccc0cdf215:0,5.0,0.986863,0.013137,Comida muito boa
2,0x94df1fccc0cdf215:0,5.0,0.986863,0.013137,Comida muito boa.
3,0x94df1fccc0cdf215:0,4.0,0.986863,0.013137,Comida muito boa!
4,0x94df1f216f80898f:0,5.0,0.986863,0.013137,A comida é muito boa
5,0x94df1ede59792de9:0,5.0,0.986863,0.013137,Comida muito boa!!!
6,0x94df18eac4b2643d:0,5.0,0.986863,0.013137,Comida muito boa!!
7,0x94df192acbd036eb:0,5.0,0.930398,0.069602,Muito boa.
8,0x94defd80e5de4f0f:0,5.0,0.930398,0.069602,Muito boa
9,0x94df18bfe8d9f45f:0,1.0,0.930398,0.069602,Muito boa



CLUSTER 13


,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df1f925f65034b:0,5.0,0.997021,0.002979,Top.
1,0x94df1f925f65034b:0,5.0,0.997021,0.002979,Top
2,0x94df1f708192624b:0,5.0,0.997021,0.002979,Top!!!
3,0x94df1ece1bfad0c7:0,5.0,0.997021,0.002979,Top!!
4,0x94df192bbbd5d5bb:0,5.0,0.997021,0.002979,Top 👍 …
5,0x94df192d88bb4431:0,5.0,0.997021,0.002979,Top 👏👏👏 …
6,0x94df1973e56bf46d:0,5.0,0.997021,0.002979,Top!
7,0x94df18bfe8d9f45f:0,5.0,0.997021,0.002979,Top 👏👏 …
8,0x94df18eb055b337f:0,5.0,0.997021,0.002979,Top !!
9,0x94df192d88bb4431:0,5.0,0.874420,0.125580,Comida top



CLUSTER 14


,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df1ee19d73e25d:0,5.0,0.963032,0.036968,Ótima comida
1,0x94df18eac4b2643d:0,5.0,0.963032,0.036968,Ótima comida!
2,0x94df192ab2980905:0,4.0,0.963032,0.036968,Comida ótima
3,0x94df192d88bb4431:0,5.0,0.963032,0.036968,A comida é ótima
4,0x94df192ab2980905:0,4.0,0.953248,0.046752,Ótima
5,0x94df18eac4b2643d:0,5.0,0.953248,0.046752,Ótima ‼️‼️‼️
6,0x94df19733e66ed3f:0,5.0,0.859793,0.140207,Ótima comida e atendimento.
7,0x94df18bfe8d9f45f:0,5.0,0.825245,0.174755,Ótima atendimento
8,0x94df18eac4b2643d:0,5.0,0.765158,0.234842,Excelente atendimento e a comida é ótima!!!!!
9,0x94defd80e5de4f0f:0,5.0,0.736400,0.263600,Ótima qualidade.


## Coesão interna

In [314]:
results_cohesion = []

for cluster_id in sorted(preprocessed_reviews_df["cluster"].unique()):

    # Índices dos documentos pertencentes ao cluster
    indices = preprocessed_reviews_df.index[preprocessed_reviews_df["cluster"] == cluster_id].tolist()

    # Extrair a submatriz de similaridade
    submatrix = similarity_tfidf[np.ix_(indices, indices)]

    # Selecionar apenas a parte superior da matriz,
    # excluindo a diagonal principal
    even_values = submatrix[
        np.triu_indices_from(submatrix, k=1)
    ]

    # Calcular a similaridade média
    mean_cohesion = even_values.mean()

    results_cohesion.append({
        "cluster": cluster_id,
        "n_documents": len(indices),
        "mean_cohesion": mean_cohesion
    })


df_coesao = pd.DataFrame(results_cohesion)

df_coesao

,cluster,n_documents,mean_cohesion
0,0,264,0.082605
1,1,997,0.010289
2,2,40,0.744782
3,3,91,0.275865
4,4,67,0.834591
5,5,168,0.128897
6,6,50,0.601422
7,7,152,0.702613
8,8,49,0.438041
9,9,58,0.680060


## Distância entre clusters

In [315]:
# Similaridade entre os centróides
similarity_centroids = cosine_similarity(
    kmeans.cluster_centers_
)

# Converter similaridade em distância
distance_centroids = 1 - similarity_centroids

# Criar DataFrame
df_dist_clusters = pd.DataFrame(
    distance_centroids,
    index=[f"Cluster {i}" for i in range(N_CLUSTERS)],
    columns=[f"Cluster {i}" for i in range(N_CLUSTERS)]
)

df_dist_clusters.round(3)

,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9,Cluster 10,Cluster 11,Cluster 12,Cluster 13,Cluster 14
Cluster 0,-0.000,0.482,0.749,0.924,0.954,0.567,0.767,0.842,0.790,0.908,0.556,0.931,0.449,0.959,0.838
Cluster 1,0.482,-0.000,0.794,0.797,0.933,0.567,0.794,0.797,0.825,0.857,0.870,0.924,0.771,0.950,0.836
Cluster 2,0.749,0.794,0.000,0.984,0.997,0.840,0.833,0.928,0.870,0.952,0.951,0.970,0.821,0.986,0.862
Cluster 3,0.924,0.797,0.984,0.000,0.979,0.935,0.971,0.970,0.975,0.986,0.984,0.991,0.969,0.983,0.985
Cluster 4,0.954,0.933,0.997,0.979,0.000,0.915,0.998,0.994,0.996,0.988,0.996,0.997,0.998,0.986,0.996
Cluster 5,0.567,0.567,0.840,0.935,0.915,0.000,0.845,0.674,0.832,0.519,0.796,0.911,0.763,0.960,0.837
Cluster 6,0.767,0.794,0.833,0.971,0.998,0.845,-0.000,0.940,0.891,0.974,0.973,0.976,0.181,0.989,0.893
Cluster 7,0.842,0.797,0.928,0.970,0.994,0.674,0.940,0.000,0.918,0.984,0.994,0.971,0.920,0.993,0.942
Cluster 8,0.790,0.825,0.870,0.975,0.996,0.832,0.891,0.918,-0.000,0.962,0.972,0.946,0.879,0.977,0.881
Cluster 9,0.908,0.857,0.952,0.986,0.988,0.519,0.974,0.984,0.962,0.000,0.978,0.971,0.963,0.992,0.966


In [316]:
results_distances = []

for i in range(N_CLUSTERS):
    for j in range(i + 1, N_CLUSTERS):

        results_distances.append({
            "cluster_1": i,
            "cluster_2": j,
            "distance": distance_centroids[i, j]
        })

df_pairs_clusters = pd.DataFrame(
    results_distances
)

df_pairs_clusters.sort_values("distance")

,cluster_1,cluster_2,distance
74,6,12,0.181068
11,0,12,0.448742
0,0,1,0.482146
63,5,9,0.518516
9,0,10,0.556187
...,...,...,...
55,4,10,0.996189
56,4,11,0.996520
28,2,4,0.997029
57,4,12,0.997949


In [317]:
# Identificar menor e maior distância entre clusters
nearest_pair = df_pairs_clusters.loc[
    df_pairs_clusters["distance"].idxmin()
]

furthest_pair = df_pairs_clusters.loc[
    df_pairs_clusters["distance"].idxmax()
]

# Clusters mais próximos
c1_prox = int(nearest_pair["cluster_1"])
c2_prox = int(nearest_pair["cluster_2"])

# Clusters mais distantes
c1_dist = int(furthest_pair["cluster_1"])
c2_dist = int(furthest_pair["cluster_2"])


print("CLUSTERS MAIS PRÓXIMOS")
print("----------------------")
print(
    f"Cluster {c1_prox} × Cluster {c2_prox}"
)
print(
    f"Distância: {nearest_pair['distance']:.3f}"
)


print("\nCLUSTERS MAIS DISTANTES")
print("-----------------------")
print(
    f"Cluster {c1_dist} × Cluster {c2_dist}"
)
print(
    f"Distância: {furthest_pair['distance']:.3f}"
)

CLUSTERS MAIS PRÓXIMOS
----------------------
Cluster 6 × Cluster 12
Distância: 0.181

CLUSTERS MAIS DISTANTES
-----------------------
Cluster 4 × Cluster 6
Distância: 0.998
